# Day15 — Enterprise AI Coding Agent / v1.0

## Goal

验证 v1.0 的结构化需求契约、固定离线评估和发布工程产物。该教程不调用 LLM、Unity、网络或 Git 写操作。

## Setup

输入仅来自当前仓库的版本模块、固定评估 fixture 和发布文件。

### Key Assumptions

- 从仓库根目录或 `day15/` 目录执行；
- Python 3.10+；
- 不需要 API Key、Unity Editor 或生成代码仓库。

In [ ]:
from pathlib import Path
import sys

current_directory = Path.cwd().resolve()
repository_root = (
    current_directory
    if (current_directory / 'project_version.py').is_file()
    else current_directory.parent
)
assert (repository_root / 'project_version.py').is_file(), '请从仓库根目录或 day15 目录运行'
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

from agents.coordinator import CoordinatorAgent
from evaluation.metrics import evaluate_suite
from evaluation.schema import load_suite
from project_version import __version__

print({'repository': repository_root.name, 'version': __version__})

## Steps

### 1. 生成确定性需求契约

Coordinator 在不增加 Agent 或模型调用的前提下，将用户目标、显式文件范围和既有质量门固化为可检查点恢复的契约。

In [ ]:
coordinator_result = CoordinatorAgent().run({
    'query': '只生成 SafeCounter.cs，不修改其他文件。',
    'agent_history': [],
})
requirement_contract = coordinator_result['requirement_contract']
assert requirement_contract['schema_version'] == 1
assert requirement_contract['scope'] == {
    'requested_files': ['SafeCounter.cs'],
    'single_file_only': True,
}
assert 'unity_compile_success' in requirement_contract['acceptance_criteria']
print(requirement_contract)

### 2. 复核固定离线基准

Day14 的五案例 fixture 保持不变；Day15 不通过修改样本来抬高指标。

In [ ]:
benchmark_suite = load_suite(repository_root / 'evaluation' / 'cases' / 'day14_benchmark.json')
evaluation_result = evaluate_suite(benchmark_suite)
offline_rates = {
    'end_to_end': evaluation_result.end_to_end_success.fraction,
    'compile': evaluation_result.compile_success.fraction,
    'repair': evaluation_result.repair_success.fraction,
    'stability': evaluation_result.functional_stability.fraction,
}
assert offline_rates == {
    'end_to_end': (2, 4),
    'compile': (2, 3),
    'repair': (1, 2),
    'stability': (5, 5),
}
print(offline_rates)

### 3. 检查发布产物一致性

版本、README、发布说明和离线 CI 必须互相一致。这里只读取文件，不修改工作区。

In [ ]:
tutorial_release_version = '1.0.0'
readme_text = (repository_root / 'README.md').read_text(encoding='utf-8')
release_text = (repository_root / 'docs' / 'releases' / 'v1.0.0.md').read_text(encoding='utf-8')
ci_text = (repository_root / '.github' / 'workflows' / 'offline-ci.yml').read_text(encoding='utf-8')
release_checks = {
    'version_badge': f'Version-v{__version__}' in readme_text,
    'release_title': f'v{tutorial_release_version}' in release_text,
    'offline_tests': 'python -m unittest discover' in ci_text,
    'compileall': 'python -m compileall' in ci_text,
    'diff_check': 'git diff --check' in ci_text,
}
assert all(release_checks.values())
print(release_checks)

## Checks

下面汇总本教程实际验证的版本、离线基准和发布文件状态。真实 Provider + Unity 证据继续引用 Day14 的独立验收记录，不进入离线指标。

In [ ]:
release_summary = {
    'version': tutorial_release_version,
    'requirement_contract_schema': requirement_contract['schema_version'],
    'offline_end_to_end': offline_rates['end_to_end'],
    'checks_passed': all(release_checks.values()),
}
print(release_summary)

## Next Steps

1. 执行完整 Python 回归、`compileall`、评估报告确定性和敏感信息审计；
2. 完成 v1.0 UI 视觉检查与代表性真实 Unity 验收；
3. 仅在维护者明确授权后创建标签、推送并发布稳定 GitHub Release。